# CS221 ViHOS — Stage F: Multi-Layer Representation Fusion + CRF (Google Colab GPU)

**Đề tài:** Phát hiện đoạn ngôn từ thù ghét và xúc phạm trong tiếng Việt (ViHOS Spans Detection)
**Tác giả:** Member 2 (Experimentation Owner)

### 1. Động lực khoa học (Từ Mục 3 — de_xuat_cai_tien_mo_hinh.md)
- Ở các Stage trước (PhoBERT Run 02 và PhoBERT+CRF Run 04), mô hình chỉ sử dụng **biểu diễn của tầng cuối cùng (Layer 12)**.
- Tầng cuối bị bão hòa bởi ngữ cảnh toàn câu, dẫn đến hiện tượng Self-Attention lan tỏa ra các từ trung tính lân cận gây lỗi **Over-span** (11.8% – 12.4%).
- Trong khi đó, tác vụ phát hiện span (Span Extraction) đòi hỏi sự kết hợp hài hòa:
  1. **Tầng 1 (Layer 1):** Nắm bắt đặc trưng từ vựng / hình thái học bề mặt (Lexical / Surface patterns) — giúp nhận diện chính xác các từ chửi bới, teencode cục bộ.
  2. **Tầng 6 (Layer 6):** Nắm bắt cấu trúc ngữ đoạn và cú pháp cục bộ (Local Phrasal / Syntactic structures).
  3. **Tầng 12 (Layer 12):** Nắm bắt ngữ nghĩa toàn câu (Global semantics / Pragmatic tone).
- **Giải pháp:** Cơ chế **Dung hợp biểu diễn Đa tầng (Multi-Layer Representation Fusion)** với trọng số học được (Learnable Softmax Weights) kết hợp tầng **Linear-Chain CRF** để kẹp chặt ranh giới BIO!

## 1. Cài đặt thư viện & Kiểm tra GPU Colab

In [ ]:
!pip install -q transformers datasets accelerate seqeval scikit-learn pandas

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("CẢNH BÁO: Hãy bật GPU trong Runtime -> Change runtime type -> T4 GPU")

## 2. Nạp dữ liệu ViHOS chính thức

In [ ]:
import os
if not os.path.exists("DATA/ViHOS_official"):
    !git clone https://huggingface.co/datasets/phusroyal/ViHOS.git DATA/ViHOS_official
    print("Đã tải thành công ViHOS official!")
else:
    print("Dữ liệu đã có sẵn trong DATA/ViHOS_official")

## 3. Cấu hình Thử nghiệm Stage F

In [ ]:
MODEL_NAME = "vinai/phobert-base-v2"
MODE = "syllable"
SELECTED_LAYERS = [1, 6, 12]      # Tầng 1 (từ vựng), Tầng 6 (cú pháp), Tầng 12 (ngữ nghĩa)
FUSION_TYPE = "weighted_sum"      # 'weighted_sum' (Learnable Softmax) hoặc 'concat'
BATCH_SIZE = 16
LEARNING_RATE_TRANSFORMER = 2e-5
LEARNING_RATE_CRF = 1e-3
LEARNING_RATE_FUSION = 5e-3       # Trọng số tầng học nhanh hơn để sớm tìm ra phân phối tối ưu
NUM_EPOCHS = 3
MAX_LENGTH = 160
SEED = 42

LABEL2ID = {"O": 0, "B-T": 1, "I-T": 2}
ID2LABEL = {0: "O", 1: "B-T", 2: "I-T"}

print(f"Cấu hình Stage F: {MODEL_NAME} | Layers={SELECTED_LAYERS} | Fusion={FUSION_TYPE}")

## 4. Định nghĩa Kiến trúc: Multi-Layer Representation Fusion + CRF

In [ ]:
import torch.nn as nn
from transformers import AutoModel, AutoConfig, AutoTokenizer
from typing import List, Optional, Tuple

class PyTorchCRF(nn.Module):
    def __init__(self, num_tags: int = 3):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        numerator = self._compute_score(emissions, tags, mask)
        denominator = self._compute_normalizer(emissions, mask)
        return -(numerator - denominator).mean()

    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        return self._viterbi_decode(emissions, mask)

    def _compute_score(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        seq_len, batch_size, _ = emissions.shape
        score = self.start_transitions[tags[0]] + emissions[0, torch.arange(batch_size), tags[0]]
        for i in range(1, seq_len):
            mask_i = mask[i]
            trans_score = self.transitions[tags[i], tags[i - 1]]
            emission_score = emissions[i, torch.arange(batch_size), tags[i]]
            score = score + (trans_score + emission_score) * mask_i
        last_valid = mask.sum(dim=0).long() - 1
        last_tags = tags[last_valid, torch.arange(batch_size)]
        score = score + self.end_transitions[last_tags]
        return score

    def _compute_normalizer(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        seq_len, batch_size, _ = emissions.shape
        forward_var = self.start_transitions + emissions[0]
        for i in range(1, seq_len):
            broadcast_emit = emissions[i].unsqueeze(2)
            broadcast_trans = self.transitions.unsqueeze(0)
            broadcast_var = forward_var.unsqueeze(1)
            next_var = torch.logsumexp(broadcast_var + broadcast_trans + broadcast_emit, dim=2)
            forward_var = torch.where(mask[i].unsqueeze(1).bool(), next_var, forward_var)
        forward_var = forward_var + self.end_transitions
        return torch.logsumexp(forward_var, dim=1)

    def _viterbi_decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        seq_len, batch_size, _ = emissions.shape
        viterbi_vars = self.start_transitions + emissions[0]
        backpointers = []
        for i in range(1, seq_len):
            broadcast_var = viterbi_vars.unsqueeze(1)
            broadcast_trans = self.transitions.unsqueeze(0)
            max_vars, bptrs = torch.max(broadcast_var + broadcast_trans, dim=2)
            max_vars = max_vars + emissions[i]
            viterbi_vars = torch.where(mask[i].unsqueeze(1).bool(), max_vars, viterbi_vars)
            backpointers.append(bptrs)
        viterbi_vars = viterbi_vars + self.end_transitions
        best_tags_list = []
        last_valid = mask.sum(dim=0).long() - 1
        for b in range(batch_size):
            last_i = last_valid[b].item()
            best_tag = torch.argmax(viterbi_vars[b]).item()
            best_tags = [best_tag]
            for i in range(last_i - 1, -1, -1):
                best_tag = backpointers[i][b, best_tag].item()
                best_tags.append(best_tag)
            best_tags.reverse()
            best_tags_list.append(best_tags)
        return best_tags_list


class MultiLayerTransformerCRF(nn.Module):
    def __init__(
        self,
        model_name: str = "vinai/phobert-base-v2",
        num_tags: int = 3,
        dropout_rate: float = 0.2,
        selected_layers: List[int] = [1, 6, 12],
        fusion_type: str = "weighted_sum"
    ):
        super().__init__()
        self.selected_layers = selected_layers
        self.fusion_type = fusion_type
        
        config = AutoConfig.from_pretrained(model_name)
        config.output_hidden_states = True
        self.transformer = AutoModel.from_pretrained(model_name, config=config)
        hidden_size = config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        
        if self.fusion_type == "weighted_sum":
            self.layer_weights = nn.Parameter(torch.ones(len(selected_layers)))
            self.classifier = nn.Linear(hidden_size, num_tags)
        elif self.fusion_type == "concat":
            self.concat_projection = nn.Linear(hidden_size * len(selected_layers), hidden_size)
            self.activation = nn.GELU()
            self.classifier = nn.Linear(hidden_size, num_tags)
        else:
            raise ValueError(f"fusion_type không hợp lệ: {fusion_type}")
            
        self.crf = PyTorchCRF(num_tags)

    def _fuse_layers(self, all_hidden_states):
        extracted = [all_hidden_states[layer_idx] for layer_idx in self.selected_layers]
        if self.fusion_type == "weighted_sum":
            normalized_weights = torch.softmax(self.layer_weights, dim=0)
            fused = torch.zeros_like(extracted[0])
            for w, h in zip(normalized_weights, extracted):
                fused = fused + w * h
            return fused
        elif self.fusion_type == "concat":
            concatenated = torch.cat(extracted, dim=-1)
            return self.activation(self.concat_projection(concatenated))

    def forward(self, input_ids, attention_mask, first_subword_indices, word_labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        all_hidden_states = outputs.hidden_states
        sequence_output = self._fuse_layers(all_hidden_states)
        sequence_output = self.dropout(sequence_output)
        
        batch_size = sequence_output.size(0)
        device = sequence_output.device
        max_words = max(len(idxs) for idxs in first_subword_indices)
        word_emissions = torch.zeros(batch_size, max_words, sequence_output.size(-1), device=device)
        word_mask = torch.zeros(batch_size, max_words, dtype=torch.uint8, device=device)
        
        for b in range(batch_size):
            idxs = first_subword_indices[b]
            num_w = len(idxs)
            if num_w > 0:
                valid_idxs = [min(idx, sequence_output.size(1) - 1) for idx in idxs]
                word_emissions[b, :num_w] = sequence_output[b, valid_idxs]
                word_mask[b, :num_w] = 1
                
        emissions = self.classifier(word_emissions)
        emissions_crf = emissions.transpose(0, 1)
        word_mask_crf = word_mask.transpose(0, 1)
        
        loss = None
        if word_labels is not None:
            padded_labels = torch.zeros(batch_size, max_words, dtype=torch.long, device=device)
            for b in range(batch_size):
                labels_b = word_labels[b]
                num_w = min(len(labels_b), max_words)
                padded_labels[b, :num_w] = torch.tensor(labels_b[:num_w], dtype=torch.long, device=device)
            padded_labels_crf = padded_labels.transpose(0, 1)
            loss = self.crf(emissions_crf, padded_labels_crf, word_mask_crf)
            
        decoded_paths = self.crf.decode(emissions_crf, word_mask_crf)
        return loss, decoded_paths

print("Đã xây dựng thành công MultiLayerTransformerCRF!")

## 5. Chuẩn bị DataLoader & Tokenization

In [ ]:
import ast
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class ViHOSSample:
    def __init__(self, sentence_id: int, content: str, tokens: List[str], tags: List[str], gold_char_offsets: set):
        self.sentence_id = sentence_id
        self.content = content
        self.tokens = tokens
        self.tags = tags
        self.gold_char_offsets = gold_char_offsets
        self.is_toxic = 1 if ("B-T" in tags or len(gold_char_offsets) > 0) else 0

def align_tokens_to_char_spans(content: str, tokens: List[str]):
    token_spans, curr_idx, content_lower = [], 0, content.lower()
    for token in tokens:
        tok_clean = str(token).strip()
        idx = content.find(tok_clean, curr_idx)
        if idx == -1: idx = content_lower.find(tok_clean.lower(), curr_idx)
        if idx != -1:
            start, end = idx, idx + len(tok_clean)
            token_spans.append((start, end))
            curr_idx = end
        else:
            start, end = curr_idx, min(curr_idx + len(tok_clean), len(content))
            token_spans.append((start, end))
            curr_idx = end
    return token_spans

def bio_to_char_offsets(tokens: List[str], tags: List[str], content: str):
    token_spans = align_tokens_to_char_spans(content, tokens)
    char_offsets, prev_end, prev_tag = set(), None, None
    for (start, end), tag in zip(token_spans, tags):
        if tag in ("B-T", "I-T"):
            if tag == "I-T" and prev_tag in ("B-T", "I-T") and prev_end is not None and prev_end < start:
                for c in range(prev_end, start): char_offsets.add(c)
            for c in range(start, end): char_offsets.add(c)
        prev_end, prev_tag = end, tag
    return char_offsets

def load_vihos_split(split: str = "train", mode: str = "syllable", data_dir: str = "DATA/ViHOS_official"):
    bio_file = f"{data_dir}/train_sequence_labeling/{mode}/{split}_BIO_syllable.csv" if mode == "syllable" else f"{data_dir}/train_sequence_labeling/{mode}/{split}_BIO_Word.csv"
    span_file = f"{data_dir}/train_span_extraction/{split}.csv" if split in ("train", "dev") else f"{data_dir}/test/test.csv"
    df_span, df_bio = pd.read_csv(span_file), pd.read_csv(bio_file)
    grouped_bio = df_bio.groupby("sentence_id")
    samples = []
    for sid in range(len(df_span)):
        span_row = df_span.iloc[sid]
        content = str(span_row["content"]) if pd.notna(span_row["content"]) else ""
        raw_spans = span_row.get("index_spans", "[]")
        gold_char_offsets = set(ast.literal_eval(str(raw_spans))) if pd.notna(raw_spans) and str(raw_spans).strip() not in ("", "nan") else set()
        if sid in grouped_bio.groups:
            grp = grouped_bio.get_group(sid)
            tokens = [str(w) if pd.notna(w) else "" for w in grp["Word"].tolist()]
            tags = [str(t) if pd.notna(t) else "O" for t in grp["Tag"].tolist()]
        else:
            tokens = content.split()
            tags = ["O"] * len(tokens)
        samples.append(ViHOSSample(sid, content, tokens, tags, gold_char_offsets))
    return samples

train_samples = load_vihos_split("train", mode=MODE)
dev_samples = load_vihos_split("dev", mode=MODE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ViHOSDataset(Dataset):
    def __init__(self, samples: List[ViHOSSample], tokenizer, max_length: int = 160):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = [self._process(s) for s in samples]

    def _process(self, sample):
        tokens, tags = sample.tokens, sample.tags
        bos = self.tokenizer.bos_token or self.tokenizer.cls_token
        eos = self.tokenizer.eos_token or self.tokenizer.sep_token
        subwords = [bos]
        first_subword_indices = []
        for word in tokens:
            w_tokens = self.tokenizer.tokenize(str(word))
            if not w_tokens: w_tokens = [self.tokenizer.unk_token]
            first_subword_indices.append(len(subwords))
            subwords.extend(w_tokens)
        subwords.append(eos)
        if len(subwords) > self.max_length:
            subwords = subwords[:self.max_length - 1] + [eos]
            first_subword_indices = [idx for idx in first_subword_indices if idx < self.max_length - 1]
        input_ids = self.tokenizer.convert_tokens_to_ids(subwords)
        pad_len = self.max_length - len(input_ids)
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 1
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids += [pad_id] * pad_len
        word_label_ids = [LABEL2ID.get(t, 0) for t in tags[:len(first_subword_indices)]]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "first_subword_indices": first_subword_indices,
            "word_labels": word_label_ids,
            "sentence_id": sample.sentence_id
        }
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.data[idx]

def collate_fn(batch):
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    first_subword_indices = [b["first_subword_indices"] for b in batch]
    word_labels = [b["word_labels"] for b in batch]
    sentence_ids = [b["sentence_id"] for b in batch]
    return input_ids, attention_mask, first_subword_indices, word_labels, sentence_ids

train_loader = DataLoader(ViHOSDataset(train_samples, tokenizer, max_length=MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(ViHOSDataset(dev_samples, tokenizer, max_length=MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"DataLoader đã sẵn sàng: Train = {len(train_loader)} batches | Dev = {len(dev_loader)} batches")

## 6. Huấn luyện Mô hình PhoBERT Multi-Layer Fusion + CRF

In [ ]:
import numpy as np
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiLayerTransformerCRF(
    model_name=MODEL_NAME,
    num_tags=len(LABEL2ID),
    selected_layers=SELECTED_LAYERS,
    fusion_type=FUSION_TYPE
).to(device)

optimizer_grouped_parameters = [
    {"params": [p for n, p in model.transformer.named_parameters()], "lr": LEARNING_RATE_TRANSFORMER},
    {"params": [p for n, p in model.classifier.named_parameters()], "lr": LEARNING_RATE_CRF},
    {"params": [p for n, p in model.crf.named_parameters()], "lr": LEARNING_RATE_CRF}
]
if FUSION_TYPE == "weighted_sum":
    optimizer_grouped_parameters.append({"params": [model.layer_weights], "lr": LEARNING_RATE_FUSION})
elif FUSION_TYPE == "concat":
    optimizer_grouped_parameters.append({"params": [p for n, p in model.concat_projection.named_parameters()], "lr": LEARNING_RATE_CRF})

optimizer = AdamW(optimizer_grouped_parameters, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

print("Bắt đầu huấn luyện Stage F trên GPU...")
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids, attention_mask, first_subword_indices, word_labels, _ = batch
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        loss, _ = model(input_ids, attention_mask, first_subword_indices, word_labels=word_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_loss = total_loss / len(train_loader)
    if FUSION_TYPE == "weighted_sum":
        norm_weights = torch.softmax(model.layer_weights, dim=0).detach().cpu().tolist()
        weights_str = ", ".join([f"L{l}: {w:.3f}" for l, w in zip(SELECTED_LAYERS, norm_weights)])
        print(f"==> Epoch {epoch} XONG! Avg Loss: {avg_loss:.4f} | Trọng số phân bổ: [{weights_str}]")
    else:
        print(f"==> Epoch {epoch} XONG! Avg Loss: {avg_loss:.4f}")

## 7. Đánh giá 5 Trụ cột trên Dev set (1,106 câu)

In [ ]:
from seqeval.metrics import precision_score as seq_p, recall_score as seq_r, f1_score as seq_f1
from sklearn.metrics import f1_score as sk_f1, accuracy_score

model.eval()
all_pred_tags = []
all_pred_offsets = []
all_pred_toxic = []

with torch.no_grad():
    sample_idx = 0
    for batch in dev_loader:
        input_ids, attention_mask, first_subword_indices, _, _ = batch
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        _, decoded_paths = model(input_ids, attention_mask, first_subword_indices)
        
        for path in decoded_paths:
            sample = dev_samples[sample_idx]
            tokens = sample.tokens
            tags = [ID2LABEL[tag_id] for tag_id in path[:len(tokens)]]
            if len(tags) < len(tokens):
                tags += ["O"] * (len(tokens) - len(tags))
            char_offsets = bio_to_char_offsets(tokens, tags, sample.content)
            is_toxic = 1 if ("B-T" in tags or len(char_offsets) > 0) else 0
            
            all_pred_tags.append(tags)
            all_pred_offsets.append(char_offsets)
            all_pred_toxic.append(is_toxic)
            sample_idx += 1

# Tính toán 5 Trụ cột đo lường chuẩn mực
gold_offsets = [s.gold_char_offsets for s in dev_samples]
gold_tags = [s.tags for s in dev_samples]
gold_toxic = [s.is_toxic for s in dev_samples]

# Trụ cột 1: SemEval Char Macro-F1
char_f1s = []
for g, p in zip(gold_offsets, all_pred_offsets):
    if not g and not p: char_f1s.append(1.0)
    elif not g or not p: char_f1s.append(0.0)
    else:
        prec = len(g.intersection(p)) / len(p)
        rec = len(g.intersection(p)) / len(g)
        char_f1s.append(2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0)
char_macro_f1 = sum(char_f1s) / len(char_f1s)

# Token BIO F1
tok_f1 = seq_f1(gold_tags, all_pred_tags)
tok_p = seq_p(gold_tags, all_pred_tags)
tok_r = seq_r(gold_tags, all_pred_tags)

# Post-level Binary F1
post_bin_f1 = sk_f1(gold_toxic, all_pred_toxic, average="binary")

# Phân rã lỗi ranh giới (Exact Match, Over-span, Under-span)
exact, over, under, miss, spurious = 0, 0, 0, 0, 0
for g, p in zip(gold_offsets, all_pred_offsets):
    if g == p and len(g) > 0: exact += 1
    elif not g and p: spurious += 1
    elif g and not p: miss += 1
    elif g and p:
        if p.issuperset(g): over += 1
        elif p.issubset(g): under += 1

n = len(dev_samples)
print("=" * 80)
print("KẾT QUẢ ĐÁNH GIÁ STAGE F (PHOBERT MULTI-LAYER FUSION + CRF):")
print("=" * 80)
print(f"  • Character Macro-F1 (SemEval):  {char_macro_f1:.4f}")
print(f"  • Token BIO F1 (Seqeval):        {tok_f1:.4f} (Precision: {tok_p:.4f}, Recall: {tok_r:.4f})")
print(f"  • Post-Level Binary-F1:          {post_bin_f1:.4f}")
print(f"  • Exact Match (Khớp 100%):       {exact / n * 100:.2f}%")
print(f"  • Over-span (Bắt thừa):          {over / n * 100:.2f}%")
print(f"  • Under-span (Bắt thiếu):        {under / n * 100:.2f}%")
print(f"  • Complete Miss (Bỏ sót):        {miss / n * 100:.2f}%")
print(f"  • Spurious Span (Báo giả):       {spurious / n * 100:.2f}%")
print("=" * 80)

## 8. Xuất file kết quả dự đoán & Tải về máy

In [ ]:
import json
from google.colab import files

records = []
for i, sample in enumerate(dev_samples):
    records.append({
        "sentence_id": sample.sentence_id,
        "content": sample.content,
        "gold_tags": " ".join(sample.tags),
        "pred_tags": " ".join(all_pred_tags[i]),
        "gold_char_offsets": str(sorted(list(sample.gold_char_offsets))),
        "pred_char_offsets": str(sorted(list(all_pred_offsets[i]))),
        "gold_toxic": sample.is_toxic,
        "pred_toxic": all_pred_toxic[i]
    })

df_out = pd.DataFrame(records)
csv_filename = "phobert_multilayer_crf_syllable_dev_predictions.csv"
df_out.to_csv(csv_filename, index=False, encoding="utf-8")
print(f"Đã xuất file dự đoán: {csv_filename} ({len(df_out)} dòng)")

summary = {
    "model": "PhoBERT_MultiLayer_Fusion_CRF",
    "selected_layers": SELECTED_LAYERS,
    "fusion_type": FUSION_TYPE,
    "Char_Macro_F1": float(char_macro_f1),
    "Token_BIO_F1": float(tok_f1),
    "Post_Binary_F1": float(post_bin_f1),
    "Exact_Match": float(exact / n * 100),
    "Over_Span": float(over / n * 100),
    "Under_Span": float(under / n * 100),
    "Complete_Miss": float(miss / n * 100),
    "Spurious_Span": float(spurious / n * 100)
}
json_filename = "phobert_multilayer_crf_syllable_summary.json"
with open(json_filename, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Đã xuất log tóm tắt: {json_filename}")

# Tự động tải về máy
files.download(csv_filename)
files.download(json_filename)